# NanoWM: watch a pretrained world model

Run the official **Point Maze NanoWM-B/2** checkpoint on recorded validation actions.
The output is an **open-loop prediction**, not an interactive game or a planning benchmark.

1. In Colab choose **Runtime → Change runtime type → GPU** (a T4 is suitable for the memory footprint; availability varies).
2. Choose **Runtime → Run all**. No dataset upload, account token, or manual restart is needed.
3. Watch all four comparisons below. **Left: recorded frames. Right: predicted frames.**

The first 3 frames are observed context; the remaining 17 are predictions. Background sharpness
alone does not establish motion accuracy: watch the green agent, especially later in each clip.
The notebook uses the repository's locked Python environment in a subprocess and leaves the
notebook kernel's packages alone. GPU, download speed, and sampling steps determine total time.

Model and VAE revisions, dataset checksum, source revision, seed, and clip identities are recorded.
The 718 MB dataset archive expands to about 30 GB in full; this demo extracts only the metadata
and the four required observation sequences. Allow at least 12 GB for the environment and assets.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

DEMO_HOME = Path(os.environ.get("NANOWM_DEMO_HOME", "/content/nanowm-demo" if Path("/content").exists() else str(Path.cwd() / "nanowm-demo"))).expanduser().resolve()
REPO_URL = os.environ.get("NANOWM_REPO_URL", "https://github.com/simchowitzlabpublic/nano-world-model.git")
REPO_REVISION = "50f5d7ec8ca0e16548e3aa602c0ec628836aa309"  # reviewed source; update intentionally when upgrading the demo
NUM_SAMPLES = 4
SAMPLING_STEPS = 50  # 15: quick preview; 50: default; 250: slower comparison, no quality guarantee
SEED = 3407
BATCH_SIZE = 1  # keeps the memory footprint small
DEMO_HOME.mkdir(parents=True, exist_ok=True)
REPO = DEMO_HOME / "repo"
ASSETS = DEMO_HOME / "assets"
OUTPUT = DEMO_HOME / f"results-{SAMPLING_STEPS}-seed-{SEED}"

def run(argv, **kwargs):
    print("Running:", " ".join(map(str, argv)), flush=True)
    subprocess.run(list(map(str, argv)), check=True, **kwargs)

run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"])
print("Workspace:", DEMO_HOME)

## 1. Install the reviewed, locked runtime

All model code runs through `./nanowm`; this does not downgrade Colab's preinstalled libraries.
Re-running cells keeps downloaded packages and assets. Use a new `DEMO_HOME` if you want an independent run.

In [ ]:
if not (REPO / ".git").exists():
    REPO.mkdir(parents=True, exist_ok=True)
    run(["git", "init", str(REPO)])
    run(["git", "remote", "add", "origin", REPO_URL], cwd=REPO)
if subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True, check=True).stdout.strip():
    raise RuntimeError("The demo checkout has local edits. Keep those edits and choose a new DEMO_HOME.")
current = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True)
if current.returncode or current.stdout.strip() != REPO_REVISION:
    run(["git", "fetch", "--depth=1", "origin", REPO_REVISION], cwd=REPO)
    run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)

bootstrap = DEMO_HOME / ".bootstrap"
uv = bootstrap / "bin" / "uv"
if not uv.exists():
    run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "--target", str(bootstrap), "uv==0.12.0"])
ENV = dict(os.environ, PATH=str(uv.parent) + os.pathsep + os.environ.get("PATH", ""))
run(["./nanowm", "sync"], cwd=REPO, env=ENV)
run(["./nanowm", "doctor", "--require-cuda", "--cuda-dtype", "float32", "--output", str(DEMO_HOME / "environment.json")], cwd=REPO, env=ENV)

## 2. Download the official model and a reproducible validation preview

The OSF archive is verified with its published SHA-256. Only selected observation files are extracted;
all trajectory metadata remains available so the train/validation split and action normalization match
the checkpoint. The checkpoint and VAE use fixed Hugging Face revisions. Preparation is safe to rerun.

In [ ]:
run(["./nanowm", "python", "scripts/prepare_colab_demo.py", "--root", ASSETS, "--num-samples", NUM_SAMPLES], cwd=REPO, env=ENV)
provenance = json.loads((ASSETS / "provenance.json").read_text())
print("Distinct validation episodes:", provenance["episodes"])

## 3. Generate predictions

Four distinct validation trajectories, in a fixed order, with the same random seed.
Increasing sampling steps costs time and does **not** guarantee a better trajectory for every clip.
Batch size affects random-number consumption; keep it unchanged when comparing runs.

In [ ]:
started = time.monotonic()
run(["./nanowm", "python", "src/sample/rollout.py",
     "--config", provenance["config"], "--ckpt", provenance["checkpoint"],
     "--save_path", OUTPUT, "--num_samples", NUM_SAMPLES, "--batch_size", BATCH_SIZE,
     "--rollout_length", 20, "--history_length", 3,
     "--num_sampling_steps", SAMPLING_STEPS, "--scheduling_mode", "sequential",
     "--history_stabilization_level", 0.02, "--fps", 8,
     "--seed", SEED, "--unique_trajectories"], cwd=REPO, env=ENV)
report = json.loads((OUTPUT / "report.json").read_text())
assert len(report["samples"]) == NUM_SAMPLES
assert [s["traj_idx"] for s in report["samples"]] == provenance["episodes"]
print(f"Finished in {time.monotonic()-started:.1f} seconds; peak model GPU allocation {report['peak_gpu_memory_gib']:.2f} GiB.")

## 4. Watch every result

**Left = recorded validation images. Right = model prediction.** The first 3 frames are context.
Play each video or scrub to the end; do not judge the moving agent from the mostly static background.
Metrics below cover only the 17 predicted frames, before video compression. The frozen-frame baseline
illustrates why high full-image PSNR/SSIM can be misleading in this small, mostly static scene.

In [ ]:
from IPython.display import display, Video, Markdown
for sample in report["samples"]:
    i = sample["sample_id"]
    display(Markdown(f"### Clip {i+1} · validation episode {sample['traj_idx']}\nRecorded (left) | predicted (right); 3 context frames + 17 predicted frames."))
    display(Video(filename=str(OUTPUT / f"sample_{i:04d}_compare.mp4"), embed=True, width=768, html_attributes="controls loop muted playsinline"))
rows = ["| Clip | Prediction PSNR / SSIM | Frozen last context PSNR / SSIM |", "|---|---|---|"]
for s in report["samples"]:
    a, b = s["prediction"], s["repeat_last_context"]
    rows.append(f"| {s['sample_id']+1} | {a['psnr_db']:.2f} dB / {a['ssim']:.4f} | {b['psnr_db']:.2f} dB / {b['ssim']:.4f} |")
display(Markdown("\n".join(rows)))

## 5. Keep the videos

Colab's temporary disk disappears when the runtime is deleted. Download the result ZIP before leaving;
it contains the videos, metrics, and reproducibility metadata. **File → Download → Download .ipynb**
also saves this notebook with its embedded video outputs. Viewing saved results needs no GPU.

In [ ]:
(OUTPUT / "provenance.json").write_text(json.dumps(dict(provenance, source_revision=REPO_REVISION), indent=2))
shutil.copy2(DEMO_HOME / "environment.json", OUTPUT / "environment.json")
archive = shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT)
try:
    from google.colab import files
except ImportError:
    print("Saved result ZIP:", archive)
else:
    files.download(archive)

## Try another setting

Change `SAMPLING_STEPS` or `SEED` in the first code cell, then rerun that cell and steps 3–5.
Changing `NUM_SAMPLES` also requires step 2 to extract the additional observations.

This is an inference preview using prerecorded actions. It does not measure planning success,
prove a universal quality improvement from more sampling steps, or reproduce the paper's benchmark.
Native simulator environments are not required. Setup errors stop the notebook immediately; rerun the
failed cell after correcting a network/GPU issue. No manual source patching or runtime restart is needed.